
#Transform circuits data
1. Read bronze table
2. keep required column for analytics
3. standardize column names using snake_case
4. rename column into meaninful names
5. filter rows with null values
6. remove duplicates
7. transform values of circuit_name and locality column to capitalize
8. write the trasnformed data to silver circuit table


In [0]:
%run ../config/config-env


In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.circuits'
silver_table = f'{catalog_name}.{silver_schema}.circuits'

In [0]:
#read bronze table
#spark.read.table(bronze_table).display()
circuits_df = spark.table(bronze_table)
display(circuits_df)

In [0]:
#keep only required columns
from pyspark.sql import functions as F
circuit_selected_df = circuits_df.select(
    F.col('circuitId'),
    F.col('circuitName'),
    F.col('locality'),
    F.col('country'),
    F.col('lat'),
    F.col('long'),
    F.col('ingestion_timestamp'),
    F.col('source_file')
)

In [0]:
# standardize column names
circuit_renamed_df = (
        circuit_selected_df
             .withColumnsRenamed({'circuitId':'circuit_id', 'circuitName':'circuit_name',
                                 'lat': 'latitude', 'long': 'longitude'
                            })
        
        )

In [0]:
# filter null data
circuit_filtered_df = circuit_renamed_df.filter(
    F.col('circuit_id').isNotNull()
    )

In [0]:
#drop duplicates based on column
circuit_valid_df= circuit_renamed_df.dropDuplicates(['circuit_id'])

In [0]:
# capitalize 

circuit_final_df = (
    circuit_valid_df
        .withColumn('circuit_name', F.initcap(F.col('circuit_name')))
        .withColumn('locality', F.initcap(F.col('locality')))
    )
display(circuit_final_df )

In [0]:
(
    circuit_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(silver_table)
)